# Introduction to MAF

<img align="left" src = https://project.lsst.org/sites/default/files/Rubin-O-Logo_0.png width=250 style="padding: 10px"> 
<b>Introduction to MAF</b> <br>
Contact authors: Eric Neilsen, Lynne Jones, Peter Yoachim<br>
Questions welcome at <a href="https://community.lsst.org/c/sci/survey-strategy">community.lsst.org/c/sci/survey-strategy</a> and the <a href="https://lsstc.slack.com/archives/C2LTWTP5J">#sims_operations</a> slack channel.<br>
Find additional MAF documentation and resources at <a href="https://rubin-sim.lsst.io">rubin-sim.lsst.io</a>. <br>

- author of corrections : Sylvie Dagoret-Campagne
- creation date : 2026-08-04
- copied and adapted (corrected) from https://github.com/lsst/rubin_sim_notebooks/tree/main/maf/tutorial
- python MAC : `kernel conda_py313_opsim53`

## 1. Basic concepts

### 1.1 Prerequisites

MAF is the "Metrics Analysis Framework" for evaluating simulations generated by the Rubin Observatory/LSST survey scheduler.

This notebook does assume some familiarity with the use of `jupyter` notebooks. The `jupyter` project has [extensive documentation](https://jupyter.org/documentation), and there are numerous tutorial videos available, including a [very short one](https://www.youtube.com/watch?v=A5YyoCKxEOU) from the `jupyter` project itself, and a [more extensive talk](https://www.youtube.com/watch?v=RFabWieskak) from SciPy 2019.

Using MAF requires the [rubin_sim](https://github.com/lsst/rubin_sim) package to be available. 

### 1.2 Essential elements

#### Purpose

The purpose of MAF is to transform the output of `opsim`, a table of visits with a variety of parameters describing each visit (time, pointing, filter, seeing, limiting magnitude, etc.) and derive plots and statistics that help evaluate the survey produced.

The basic sequence looks like this:

![topdfd](./figures/topdfd.png)

#### Metric values and slices

Plots used to evaluate simulated strategies often map two kinds of data onto features in the figure:
 - Quantities that describe a subset of the visits. For example, the location and width of a bar in a histogram represent the definition of the subset of visits that are included in the bar. In a map of the sky, the location of a pixel represents a subet of visits (those covering that pixel).
 - Quantities derived from members of the subset. For example, the height of a bar in a histogram is typically a count of the number of visits in the bar, and in a sky map the color of a pixel in the map represents some function of the visits that cover that pixel.
 
MAF uses the concept of slices and metrics to represent the subset definitions and quantities derived from those subsets, respectively:
 - a slicer (instance of a subclass of `rubin_sim.maf.slicers.BaseSlicer`) defines a set of subsets ("slices") and figures out which visits fall in which slices.
 - a metric (instance of a subclass of `rubin_sim.maf.metrics.BaseMetric`) computes values for each slice using the slice point definitions (e.g. bin limits or coordinates on the sky) and visits assigned to each slice.
 
MAF saves slice definitions and metric values to disk, and these can be reloaded later for additional exploration.

The process of dividing the visits into subsets and computing values for each subset resembles a traditional "split/apply/combine" pattern, as implemented in SQL queries, or [pandas `groupby`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html) operations: slicers implement something like an SQL "GROUP BY" operation, while the metrics themselves resemble SQL aggregation operations. MAF's slicers differ from a traditional "split" in one important respect, however: in some slicers, a visit might fall in multiple slices. For example, a single visit on the sky can cover multiple healpixels on the sky, and so fall in multiple spatial slices.

When computing metrics, one often wants to limit the visits that are included. For example, one might wish to consider only visits in a given filter, or include only visits that are deeper than some magnitude. So, the three elements are needed to define the transformation from the database of visits to the resultant metric values and slice points:
 - a constraint, a fragment of SQL that limits the visits to be included.
 - a slicer, a python class that computes slice parameters and computes which visits are included in each slice.
 - a metric, a python class that computes a value (often a scalar, put potentially any python object) for each slice.

#### Plots and summary statistics

The slices and metric values themselves are generally not easily digestable. So, two additional steps are performed:
- Plots can be made, mapping metric values and slice point parameters to features in the plot. A given slicer and metric may result in multiple plots. For example, a spatial metric might both a map and a histogram of coadd depth.
- Summary stastics can be computed. Summary statistics do not vary by slice point, but rather are single scalars that apply to the survey as a whole. Individual metrics may have multiple summary statistics. For example, summary statistics for a metric mapping the depth over the footprint can have median depth (over the footprint), minimum depth, maximum depth, and other statistics over the whole survey.

#### Basic elements of the MAF infrastructure

The basic elements of the MAF architecture that a user needs to understand are:

| element | base class | description |
|:------- |:---------- |:----------- |
| database | `str` | The database provides access to an sqlite database of visits, usually generated by the operations simulater, opsim. In current versions of rubin_sim, the "database" is simply the name of the database, and is thus just a string. |
| constraint | `str` | The constraint is the content of an SQL `WHERE` clause that selects which visits from the visit database should be included in the calculation metrics. This can be `None`.|
| slicer | `rubin_sim.maf.slicers.BaseSlicer` | A *slicer* divides visits into subsets, similar to the SQL `GROUP BY` clause or the `pandas.DataFrame.groupby` method. Every MAF *slicer* is a python iterator. Each item returned by the iterator is a dictionary with two elements: `idxs`, a `numpy.array` of visit indices that fall in the slice; and `slicePoint`, which indicates the parameters of the slice. The value of the `slicePoint` depends on the specific slicer. For example, the `OneDSlicer`, which divides visits up into bins by a value (as for a histogram), returns `slicePoint`s that are dictionaries specifying the number and location of each bin; while a `HealpixSlicer` returns `slicePoint`s that specify the healpix nside used, the healpix id, and its RA and declination.
| metric | `rubin_sim.maf.metrics.BaseMetric` | A *metric* calculates a values for each *slice* returned by iterating over `slicer`. Each metric class defines a `run` method that takes a `numpy.recarray` of visits and a (perhaps optional) `slicePoint`, and returns a result, typically but not always scalar. For example, the `maxMetric` returns the maximum value of some parameter (specified in initialization of the metric) taken by visits in the supplied `numpy.recarary` of visits. Many metrics depend only on the visits in each slice, and can be used with arbitrary slicers. A few, though, use data in the `slicePoint`, and so may only be used with *slicers* whose `slicePoint`s include the data they require.
| plotter | `rubin_sim.maf.plots.BasePlotter` | A *plotter* builds `matplotlib.figure.Figure` that represents the `slicePoints` and values computed my *metrics* graphically. A *plotter* is a python callable object which takes as paramters the metric values (computed by a *metric*) and a *slicer*. Figure parameters can be customized using an optional `userPlotDict` parameter. *Slicers* often have a default set of plotters, so users may not need to specify them.
| metric bundle | `rubin_sim.maf.metricBundles.MetricBundle` | An instance of a *metric bundle* combines an instance of a constraint (which is allowed to be an empty string), a slicer, a metric, and (optionally) a list of plotters to form a set. If the plotters are not specified explicitly, the default set specified by the *slicer* are used. **The *metric bundle* is the primary unit of work in MAF,** containing the elements needed to specify how a metric is to be computed and presented, and storing the results and metadata (e.g. the instance of the *slicer* with its *slice points*) needed to interpret them.  
| metric bundle group | `rubin_sim.maf.metricBundles.MetricBundleGroup` | A *metric bundle group* combines a *database*, a dictionary of *metric bundles*, and (optionally) a database and directory in which to store results. The `MetricBundleGroup.run_all` method queries the database for the necessary data, calculates metrics using each bundle group, and saves the results both within the `MetricBundle` instances and on disk. The `MetricBundleGroup.plot_all` method runs all plotters in all metric bundles. The purpose of a *metric bundle group* is to efficiently run *metric bundles*, "filling out" the bundles with metric values and computed slice points, while avoiding duplication of computations when multiple bundle groups have common elements (e.g. *slicers* and database queries).  

### 1.3 A typical MAF workflow

A typical workflow for calculating MAF metrics on a single run has the following stages:
1. Identify the sqlite database that records the visits created by an `opsim` simulation. (as of recent versions of `rubin_sim`, we just use the name of the sqlite database file containing the opsim output pointings).
2. Create a set of *metric bundles*. To create a *metric bundle*:
 * Define a *constraint* to limit which visits from the simulation are to be queried from the database.
 * Instantiate a *slicer* object to define "slices," subsets of visits. The same visits may appear in multiple subsets. If the metric to be used does use subsets, create a `UniSlicer`, which creates a single slice containing all visits (subject to the defined constraint).
 * Instantiate a *metric* object to specify the computations to be performed on each slice.
 * (optional) Create a list of instances of *plotter* objects to specify what visualizations should be made.
 * (optional) Define a `plot_dict` to specify parameters refining the appearance of the plotters.
 * Instantiate the `MetricBundle` object.
3. Put your *metric bundles* into a dictionary (or list), and run them all using a `MetricBundleGroup`. The *metric bundle group* will work through each *metric bundle* in the dictionary and calculate all the needed metric values and slice points, and update the *metric bundle* objects in the dictionary with them.
4. The `plot` method of each *metric bundle* can be used visualise each *metric bundle* individually, or the `plot_all` method of the *metric bundle group* can be used to create plots for all *metric bundles* in the group.
5. Visualizations for a *metric bundle* can be supplemented and refined using the `set_plot_funcs` and `set_plot_dict` methods of the instancs of `MetricBundle`. Alternately, the metric values and slice points in a *metric bundle* can be used directly and visualized using arbitrary python visualization libraries like `matplotlib`.

## 2. Notebook preliminaries

### 2.1 Installing MAF

MAF is part of the `rubin_sim` product. Instructions for using conda to install `rubin_sim` can be found in the README of the the `rubin_sim` [github product](https://github.com/lsst/rubin_sim).

In [ ]:
import os
# on my MAC
# os.environ["RUBIN_SIM_DATA_DIR"] = "/users/dagoret/DATA/OpSim"

In [ ]:
path_topdir = os.getenv("RUBIN_SIM_DATA_DIR")
print(f"path_topdir = {path_topdir}")

### 2.2 Developer aids

The following is a development style aid; only uncomment if developing the notebook:

In [ ]:
# %load_ext lab_black
# %load_ext pycodestyle_magic
# %flake8_on --ignore E501,W505

### 2.3 Import required python modules

This tutorial requires MAF itself, which can be imported thus:

In [ ]:
from rubin_sim import maf

from rubin_sim.data import get_data_dir

try:
    from rubin_sim.data import get_baseline
except ImportError:
    from rubin_scheduler.data import get_baseline

In [ ]:
from rubin_sim.data import get_data_dir

print("get_data_dir() : ", get_data_dir())

Show which version of MAF this notebook was last run with:

In [ ]:
import rubin_sim

rubin_sim.__version__

### 2.4 Set the storage directory for this notebook

This notebook will produce output files. By default (if `data_dir = None` in the cell below), the notebook will create a temporary directory for them. If a temporary directory is created, it will automatically be deleted when the notebook kernel is stopped or restard. If you want to keep the output, set `data_dir` in the cell below to the directory where you want to keep the output.

In [ ]:
data_dir = None

To use the current local directory as the `data_dir`, uncomment this cell:

In [ ]:
# data_dir = "."

Now, if we have not customized our `data_dir`, create a temporary directory to use for this notebook. **Note that all data in this temporary directory will be deleted when the notebook kernel is stopped or restarted.**

In [ ]:
if data_dir is None:
    import tempfile
    import os

    data_dir_itself = tempfile.TemporaryDirectory(prefix="01_intro_to_maf_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")

### 2.5 Get a database to work with

To calculate metrics, we first need the data from simulations to run metrics on.

The installation of `rubin_sim` includes the opsim database for the baseline run. Let`s use that as our example:

In [ ]:
!ls /users/dagoret/DATA/OpSim

In [ ]:
# import os
# os.environ["RUBIN_SIM_DATA_DIR"] = "/users/dagoret/DATA/OpSim"

In [ ]:
# from rubin_sim.data import rs_download_testing
# rs_download_testing()

#### Extract the database name

- Note the `get_baseline()` function add a dir `sim_baseline/` to `"RUBIN_SIM_DATA_DIR"`

In [ ]:
opsim_fname = get_baseline()
opsim_fname

#### Example of dumping the database directly with sqlite3

In [ ]:
import sqlite3

conn = sqlite3.connect(opsim_fname)
cursor = conn.cursor()

#####  Find the tables

In [ ]:
cursor.execute(
    """
SELECT name FROM sqlite_master WHERE type='table'
"""
)
cursor.fetchall()

##### Find the columns of observations Table

In [ ]:
cursor.execute("PRAGMA table_info(observations)")
columns = cursor.fetchall()
# view the index, name and type of columns
# for col in columns:
#    print(col)
column_names = [row[1] for row in columns]
print(column_names)

##### Find the name of filters

In [ ]:
cursor.execute(
    """
SELECT DISTINCT filter
FROM observations
"""
)
cursor.fetchall()

##### Find the name of bands

In [ ]:
cursor.execute(
    """
SELECT DISTINCT band
FROM observations
"""
)
cursor.fetchall()

##### Transfer the database into a pandas dataframe

In [ ]:
import pandas as pd

df = pd.read_sql("SELECT * FROM observations LIMIT 5", conn)

df.head()

##### Close the database

In [ ]:
# close the connection
conn.close()

### Other info from Name

It's also helpful to have a run name, a string that labels our simulation. Let's build one from the file name:

In [ ]:
from os.path import splitext, basename

run_name = splitext(basename(opsim_fname))[0]
run_name

## 3. A detailed example: the histogram of airmass of visits in *g*

### 3.1 Computing the metric

In this simple example, we will create a histogram of the airmass of visits in *g* band.

#### Identify the opsim sqlite database:

In [ ]:
example1_opsim_db = opsim_fname
example1_opsim_db

In [ ]:
!ls -l /users/dagoret/DATA/OpSim/sim_baseline/baseline_v5.3.0_10yrs.db

#### Specify that we want to consider only *g* band visits using the *constraint*:

In [ ]:
# example1_constraint = "filter = 'g_6'"
example1_constraint = "band = 'g'"

#### Then create a slicer that "slices" the visits by airmass:

In [ ]:
example1_slicer = maf.OneDSlicer(slice_col_name="airmass", bin_min=1.0, bin_max=2.5, bin_size=0.05)

#### Create the metrics : 

- We just want to count the number of visits in each bin:

In [ ]:
example1_metric = maf.CountMetric(col="airmass")

#### Metric bundle

- Now, we group these together in a metric bundle.
- (We do not need to specify a plotter, because `maf.OneDSlicer` provides a suitable one by default.)

In [ ]:
example1_bundle = maf.MetricBundle(
    example1_metric,
    example1_slicer,
    example1_constraint,
    run_name=run_name,
)

The `run_name` is optional, but useful for constructing file names and storing metadata in output.

#### MetricBundle Group

- Combine the metric bundle with the database to form a *metric bundle group*.
- Furthermore, let's use the `out_dir` parameter to specify that the results be stored in our `data_dir`.

In [ ]:
example1_bg = maf.MetricBundleGroup([example1_bundle], example1_opsim_db, out_dir=data_dir)

#### Actually calculate the metrics:

In [ ]:
example1_bg.run_all()

### 3.2 Making a plot

In [ ]:
example1_bg.plot_all(closefigs=False)

Be default, `maf.MetricBundleGroup.plot_all` saves plots as pdf files in the current working directory. 

The save directory can be changed by setting the `out_dir` parameter when creating the `MetricBundleGroup`, and the output format by setting `figformat` parameter when calling its `plot_all` method.

### 3.3 Using metric values and slice points directly

We are not limited to creating plots. The *metric bundle* objecs provide access to the metric values and *slice points* themselves, allowing us to use any python tools we like for further analysis or visualization.

These are typically in the form of a `numpy.ma.MaskedArray`:

In [ ]:
example1_bundle.metric_values

Understanding the values in `MetricBundle.metric_values` often depends on knowing the *slice points* used. In this example, knowing the numbers of visits in each bin is of little help without knowledge of where the bins are. We can get this information by looking directly at the slicer:

In [ ]:
example1_bundle.slicer.slice_points

In this case, the `example1_bg.bundleDict['Airmass histogram (g)'].slicer.slicePoints['bins']` gives the bin edges, allowing us to determine which metric values correspond to which bins.

In [ ]:
# le bundle group est une liste de bundles
# example1_bundle.bundleDict['Airmass histogram (g)'].slicer.slicePoints['bins']

### 3.4 Saved results

In addition to providing access to the plots and metric and slice values through the python interpreter, the *metric bundle group* also saved the plots and values in the `outDir` we provided:

In [ ]:
!ls -alth $data_dir

- We can see that the plot we just made is present as a `pdf` file.
- The `npz` file contains the metric and slicer values, from which we can recreate our metric bundle:

### 3.5 Reload MAF results

In [ ]:
from os import path

example1_reloaded = maf.MetricBundle.load(
    path.join(
        data_dir,
        run_name.replace(".", "_") + "_Count_airmass_g_ONED.npz",
    )
)

The metric values are now reloaded:

In [ ]:
example1_reloaded.metric_values

as is the slicer with its slice points

In [ ]:
example1_reloaded.slicer

In [ ]:
example1_reloaded.slicer.slice_points

and further information regarding how the metric, slicer, or sql constraint were set up:

In [ ]:
example1_reloaded.info_label

### 3.6 Plot customization

If you want to customize the appearance of the plot, you can set `plotDict` when creating the `MetricBundle`, for example:

In [ ]:
example1a_metric = maf.CountMetric(col="airmass")
example1a_slicer = maf.OneDSlicer(slice_col_name="airmass", bin_min=1.0, bin_max=2.5, bin_size=0.05)
example1a_bundle = maf.MetricBundle(
    example1a_metric,
    example1a_slicer,
    example1_constraint,
    run_name=run_name,
    plot_dict={"color": "r"},
)
example1a_bg = maf.MetricBundleGroup([example1a_bundle], example1_opsim_db)
example1a_bg.run_all()
example1a_bg.plot_all(closefigs=False)

## 4. Comparing two simulations

To compare two simulations, we need a second one to compare our first one with.

If you already have one, set it here:

In [ ]:
from os import path

other_opsim_fname = path.join(os.getenv("RUBIN_SIM_DATA_DIR"), "faster_templates_v5.3.0_10yrs.db")

and assign it a run name, just as we did with our baseline simulation above:

In [ ]:
from os.path import splitext, basename

other_run_name = splitext(basename(other_opsim_fname))[0]
other_run_name

Repeat the process show in example 1 to calculate metrics for the other example database:

In [ ]:
# example2_constraint = "filter = 'g_6'"
example2_constraint = "band = 'g'"

example2_slicer = maf.OneDSlicer(slice_col_name="airmass", bin_min=1.0, bin_max=2.5, bin_size=0.05)
example2_metric = maf.CountMetric(col="airmass")
example2_bundle = maf.MetricBundle(
    example2_metric,
    example2_slicer,
    example2_constraint,
    run_name=other_run_name,
)
example2_bg = maf.MetricBundleGroup([example2_bundle], other_opsim_fname, out_dir=data_dir)
example2_bg.run_all()

We can use a *plot handler* to combine plots of the first and second databases:

In [ ]:
example2_ph = maf.PlotHandler(out_dir=data_dir)

example2_ph.set_metric_bundles([example1_bundle, example2_bundle])
plot_dicts = [
    {"label": run_name, "color": "b"},
    {"label": other_run_name, "color": "r"},
]

fig = example2_ph.plot(plot_func=maf.OneDBinnedData(), plot_dicts=plot_dicts)

## 5. A metric that maps the sky: depth in *r*

Making maps follows a similar procedure, but uses a slicer that slices based on position in the sky, usually `rubin_sim.maf.slicers.HealpixSlicer`.
`HealpixSlicer` creates a slice for each [healpixel](https://arxiv.org/abs/astro-ph/9905275), and each slice contains all visits that over that slice's healpixel.

First, lets sets the constraint only to look at visits in *r*:

In [ ]:
# example3_constraint = "filter = 'r_57'"
example3_constraint = "band = 'r'"

To make a map, we need to slice the visits by pointing on the sky. Note that the same visit may appear in multiple slices, so this will work correctly even in the resolution of the map is smaller than the camera footprint.

In [ ]:
example3_slicer = maf.HealpixSlicer(nside=64)

We want to map the coadd depth, so we select a metric that estimates the coadd depth from all visits in a slice:

In [ ]:
example3_metric = maf.Coaddm5Metric()

Now, we group these together in a metric bundle, create a metric bundle group, and calculate the metrics just as we did in example 1. Note that we are reusing the same database we used in example 1.

In [ ]:
example3_bundle = maf.MetricBundle(
    example3_metric,
    example3_slicer,
    example3_constraint,
    run_name=run_name,
)
example3_bg = maf.MetricBundleGroup([example3_bundle], example1_opsim_db, out_dir=data_dir)
example3_bg.run_all()

In [ ]:
example3_bg.plot_all(closefigs=False)

Again, the values of the metric and slice point can be used directly:

In [ ]:
example3_bundle.metric_values

The *slice points* themselves are expressed using a different structure, reflecting the different way in which the visits were sliced:

In [ ]:
example3_bundle.slicer.slice_points

Note that, for the `maf.HealpixSlicer` and `maf.HealpixSubsetSlicer`, the `sid` is the `healpix` index (`ipix` in the [healpy documentation](https://healpy.readthedocs.io/en/latest/)), and `ra` and `dec` are in radians, not degrees. 

## 6. Plot customization after initial plotting

Plots may be customized after the initial execution through the bundle group, without recomputation of the metric. This is done by using the `plot` method of the metric bundle itself.

For example, you can adjust the parameters of the Mollweide projection to put the south pole at the center, such that the footprint is in the low distortion area at the center of the projection, and the area near the north pole at the high distortion area on the far left and right:

In [ ]:
example3_bundle.set_plot_dict({"rot": (0, -90, 0)})
example3_bundle.set_plot_funcs([maf.HealpixSkyMap()])
example3_bundle.plot()

You can, of course, also use the `PlotHandler` again. Note that changes you made directly to the bundle's plot_dict are kept until they are reset. Changes passed to the `PlotHandler` `plot_dicts` kwarg are temporary, for that plot only. 

In [ ]:
ph = maf.PlotHandler(out_dir=data_dir, thumbnail=False, fig_format="png")

ph.set_metric_bundles([example3_bundle])
plot_dict = {"color_min": 23, "color_max": 28, "extend": "both"}

fig = ph.plot(plot_func=maf.HealpixSkyMap(), plot_dicts=plot_dict)

In [ ]:
ph.set_metric_bundles([example3_bundle])
plot_dict = {"rot": (0, 0, 0)}

fig = ph.plot(plot_func=maf.HealpixSkyMap(), plot_dicts=plot_dict)

## 7. Summary statistics

In addition to creating figures, MAF can calculate "summary statistics" using summary metrics. These are any `Metric` which is run on the bundle's metric_values themselves, instead of the data slices (opsim) values.  Simple metrics such as `MaxMetric`, `MeanMetric`, `MinMetric` are commonly used; however **these summary metrics could be more complex, such as calculating the 3x2ptFoM summary statistic** after calculating the extragalactic coadded depth.

For example, to get statistics on the coadd depth over the sky, we can add summary metrics to example 3.

First, make a list of the metrics, define a new metric bundle that includes them, and create a bundle group to actually calculate the values of the metrics:

In [ ]:
example3a_slicer = maf.HealpixSlicer(nside=64)
example3a_metric = maf.Coaddm5Metric()
example3a_summary_metrics = [
    maf.MinMetric(),
    maf.MedianMetric(),
    maf.MaxMetric(),
    maf.RmsMetric(),
]
example3a_bundle = maf.MetricBundle(
    example3a_metric,
    example3a_slicer,
    example3_constraint,
    summary_metrics=example3a_summary_metrics,
    run_name=run_name,
)
example3a_bg = maf.MetricBundleGroup([example3a_bundle], example1_opsim_db, out_dir=data_dir)

Then, use the bundle group to drive the calculation of the metrics and summary metrics:

In [ ]:
example3a_bg.run_all()
example3a_bg.summary_all()

If you have the metric values already calculated and then need to add a summary metric, this can be done by adding these additional summary metrics, then calling the appropriate function on the metric bundle: 

In [ ]:
# Add the mean summary value
summary_addon = maf.MeanMetric()
example3a_bundle.set_summary_metrics(summary_addon)

example3a_bundle.compute_summary_stats()

Finally, look at the summary values:

In [ ]:
example3a_bundle.summary_values

## 8. Working with multiple metrics on the same simulation at once

As implied by the name "metric bundle group", multiple metric bundles can be combined into the some group and calculated "in one go".

Start by creating two new *metric bundles*:

In [ ]:
example4_bundles = {
    "g": maf.MetricBundle(
        metric=maf.CountMetric(col="airmass"),
        slicer=maf.OneDSlicer(slice_col_name="airmass", bin_min=1.0, bin_max=2.5, bin_size=0.05),
        # constraint="filter = 'g_6'",
        constraint="band = 'g'",
        plot_dict={"color": "g"},
        run_name=run_name,
    ),
    "r": maf.MetricBundle(
        metric=maf.CountMetric(col="airmass"),
        slicer=maf.OneDSlicer(slice_col_name="airmass", bin_min=1.0, bin_max=2.5, bin_size=0.05),
        # constraint="filter = 'r_57'",
        constraint="band = 'r'",
        plot_dict={"color": "r"},
        run_name=run_name,
    ),
}

Now add them both to the same *bundle group*:

In [ ]:
example4_bg = maf.MetricBundleGroup(example4_bundles, example1_opsim_db, out_dir=data_dir)
example4_bg.run_all()
example4_bg.plot_all(closefigs=False)

## 9. Finding available database columns with which to express constraints, slices, and metrics -- and STACKERS

Documentation on the contents of modern opsim output databases can be found in the `rubin_scheduler` [documentation](https://rubin-scheduler.lsst.io) at https://rubin-scheduler.lsst.io/fbs-output-schema.html

In [ ]:
import sqlite3
import pandas as pd

db = sqlite3.connect(opsim_fname)
table = pd.read_sql_query("SELECT * from observations limit 2", db)
table.columns

In addition to the columns present in the database, a number of quantities derived from them can be automatically calculated "on the fly" and used as if they were database columns. MAF calculates thes derived parameters using *stackers*. 

For example, hour angle is not a column in the database, but MAF includes the `rubin_sim.maf.HourAngleStacker`, and so `HA` can be used as if it were a column:

In [ ]:
# example5_constraint = "filter = 'g_6'"
example5_constraint = "band = 'g'"
example5_slicer = maf.OneDSlicer(slice_col_name="HA", bin_min=-12.0, bin_max=12, bin_size=0.5)
example5_metric = maf.CountMetric(col="HA")
example5_bundle = maf.MetricBundle(
    example5_metric,
    example5_slicer,
    example1_constraint,
    run_name=run_name,
)
example5_bg = maf.MetricBundleGroup([example5_bundle], example1_opsim_db, out_dir=data_dir)
example5_bg.run_all()
example5_bg.plot_all(closefigs=False)

You can find the available *stackers* in the [`rubin-sim` documentation](https://rubin-sim.lsst.io/maf-api-stackers.html) or just list them within python along with the columns they create like this: 

In [ ]:
import pprint

pprint.pprint({k: v.cols_added for k, v in maf.BaseStacker.registry.items()})

## 10. Finding more documentation

More documentation on available `Metrics`, `Slicers` and `Stackers` can be found in the MAF [API documentation](https://rubin-sim.lsst.io/api.html).  A short list of all available metrics can also be found in the [`rubin-sim` documentation](https://rubin-sim.lsst.io/maf-metric-list.html), with more information in the [API documentation](https://rubin-sim.lsst.io/api.html).


Python's help can also be used for on-the-fly docstring inspection. 

The `BaseMetric`, `BaseSlicer` and `BaseStacker` classes also carry information on available metrics, slicers, and stackers, via a registry:

In [ ]:
list(maf.BaseSlicer.registry.keys())

In addition, there is a built-in "help" function for these base classes that will list all of the contents of the registry and with a `doc=True` kwarg, also show a summary of the docstring.

In [ ]:
maf.BaseMetric.help(doc=False)

**Acknowledgements:** These tutorial notebooks have benefited from previous work in MAF tutorials, including not only the [tutorial notebooks in sims_MAF-contrib](https://github.com/LSST-nonproject/sims_maf_contrib/tree/master/tutorials) but also those by  Weixiang Yu, Gordon Richards, and Will Clarkson in their [LSST_OpSim](https://github.com/RichardsGroup/LSST_OpSim) repository, inspired the material to be included here. Stylistic elements of these notebooks were guided by the DP0.1 notebooks developed by Melissa Graham and the Rubin Observatory Community Engagement Team.